In [25]:
from monai.networks.nets import UNet
from monai.networks.layers import Norm
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [26]:
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm=Norm.BATCH
).to(device)
model.eval()

print(model)

model.load_state_dict(torch.load("best_metric_model_new.pth"))

UNet(
  (model): Sequential(
    (0): ResidualUnit(
      (conv): Sequential(
        (unit0): Convolution(
          (conv): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
          (adn): ADN(
            (N): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (D): Dropout(p=0.0, inplace=False)
            (A): PReLU(num_parameters=1)
          )
        )
        (unit1): Convolution(
          (conv): Conv3d(16, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
          (adn): ADN(
            (N): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (D): Dropout(p=0.0, inplace=False)
            (A): PReLU(num_parameters=1)
          )
        )
      )
      (residual): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
    )
    (1): SkipConnection(
      (submodule): Sequential(
        (0): ResidualUnit(
          (conv): Sequential(


<All keys matched successfully>

In [27]:
from dataloading.load_atlas import load_atlas

train_id_loader, val_id_loader, test_id_loader = load_atlas()

Found 655 image files
Found 655 mask files
Found 300 test image files
Train data: 524 files
Validation data: 131 files


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


In [28]:
batch = next(iter(train_id_loader))
image = batch["image"].to(device)
label = batch["label"].to(device)
print(f"Image shape: {image.shape}, Label shape: {label.shape}")


Image shape: torch.Size([1, 1, 208, 240, 192]), Label shape: torch.Size([1, 1, 208, 240, 192])


In [29]:
import torch.nn.functional as F

def get_penultimate_features(model, image):
    with torch.no_grad():
        image = F.interpolate(image, size=(96, 96, 96), mode='trilinear', align_corners=False)
        features = model.model[0](image)
        features = model.model[1](features)
        return features

features = get_penultimate_features(model, image)
features.shape

torch.Size([1, 32, 48, 48, 48])

In [30]:
import torch.nn.functional as F

CLASSES = [0, 1]

class GaussianMeanCalculator:
    def __init__(self):
        self.class_accumulator = {c: torch.zeros(32, dtype=torch.float64).to(device) for c in CLASSES}
        self.class_counts = {c: 0 for c in CLASSES}
    
    def update(self, image, label):
        features = get_penultimate_features(model, image)
        label_interpolated = F.interpolate(label, size=(features.shape[2:5]), mode='nearest').long()

        features_flat = features.permute(0, 2, 3, 4, 1).reshape(-1, 32)
        labels_flat = label_interpolated.reshape(-1)

        for c in CLASSES:
            self.class_accumulator[c] += features_flat[labels_flat == c].sum(dim=0)
            self.class_counts[c] += (labels_flat == c).sum().item()
    
    def compute_means(self):
        means = {c: self.class_accumulator[c] / self.class_counts[c] for c in CLASSES}
        return means

In [31]:
from tqdm import tqdm

def calculate_global_class_means(loader):
    calculator = GaussianMeanCalculator()

    for batch in tqdm(loader):
        image = batch["image"].to(device)
        label = batch["label"].to(device)
        calculator.update(image, label)

    means = calculator.compute_means()
    return means

class_means = calculate_global_class_means(train_id_loader)
print(class_means)

100%|██████████| 524/524 [01:27<00:00,  5.97it/s]

{0: metatensor([ 0.1746,  0.4833, -0.0784,  0.2894,  0.4620,  0.4064,  0.4019,  0.4328,
         0.4544,  0.2593,  0.2166,  0.2083,  0.1062, -0.1535,  0.0950,  0.5071,
         0.7925,  0.5248,  0.5357,  0.3315,  0.6233,  0.6533,  0.5683,  0.5783,
         0.2206,  0.7548,  0.7472,  0.2550,  0.3797,  0.5046,  0.6338,  0.7191],
       device='cuda:0', dtype=torch.float64), 1: metatensor([ 0.0033,  0.4307, -0.1496,  0.2721,  0.3774,  0.0225,  0.3181,  0.4824,
         0.2539,  0.1036,  0.1471, -0.0131,  0.0470, -0.2448,  0.1535,  0.3910,
         0.5762,  1.0353,  1.1238,  0.9848,  1.1167,  0.5591,  0.7374,  0.7768,
         1.0553,  0.6417,  0.6499,  1.1613,  0.7917,  0.7837,  0.8017,  0.6974],
       device='cuda:0', dtype=torch.float64)}


In [32]:
class GaussianTiedCovarianceCalculator:
    def __init__(self, class_means):
        self.class_means = class_means
        self.cov_accumulator = torch.zeros(32, 32, dtype=torch.float64).to(device)
        self.total_count = 0
    
    def update(self, image, label):
        features = get_penultimate_features(model, image)
        label_interpolated = F.interpolate(label, size=(features.shape[2:5]), mode='nearest').long()

        features_flat = features.permute(0, 2, 3, 4, 1).reshape(-1, 32)
        labels_flat = label_interpolated.reshape(-1)

        for c in CLASSES:
            class_features = features_flat[labels_flat == c]
            mean = self.class_means[c].unsqueeze(0)
            diffs = class_features - mean
            self.cov_accumulator += diffs.t() @ diffs
            self.total_count += class_features.size(0)
    
    def compute_covariance(self):
        covariance = self.cov_accumulator / self.total_count
        return covariance

In [33]:
def calculate_global_tied_covariance(loader, class_means):
    calculator = GaussianTiedCovarianceCalculator(class_means)

    for batch in tqdm(loader):
        image = batch["image"].to(device)
        label = batch["label"].to(device)
        calculator.update(image, label)

    covariance = calculator.compute_covariance()
    return covariance

covariance = calculate_global_tied_covariance(train_id_loader, class_means)
print(covariance)

100%|██████████| 524/524 [01:28<00:00,  5.92it/s]

metatensor([[ 0.2211, -0.0043,  0.0145,  ...,  0.0015, -0.0089,  0.0221],
        [-0.0043,  0.1435, -0.0171,  ...,  0.0009, -0.0076,  0.0177],
        [ 0.0145, -0.0171,  0.1478,  ...,  0.0084,  0.0190, -0.0030],
        ...,
        [ 0.0015,  0.0009,  0.0084,  ...,  0.2266,  0.1199,  0.0149],
        [-0.0089, -0.0076,  0.0190,  ...,  0.1199,  0.4881,  0.0742],
        [ 0.0221,  0.0177, -0.0030,  ...,  0.0149,  0.0742,  0.2252]],
       device='cuda:0', dtype=torch.float64)


In [34]:
def calculate_mahalanobis_confidence_score(features, class_means, covariance):
    features_flat = features.permute(0, 2, 3, 4, 1).reshape(-1, 32)
    inv_cov = torch.linalg.pinv(covariance + 1e-6 * torch.eye(covariance.size(0)).to(device))
    scores = []
    for c in CLASSES:
        mean = class_means[c]
        diffs = features_flat - mean
        tmp = diffs @ inv_cov
        score = torch.sum(tmp * diffs, dim=1)
        scores.append(-score)
    scores = torch.stack(scores, dim=0)
    max_scores = torch.max(scores, dim=0).values
    return max_scores.reshape(features.shape[2:])

torch.backends.cuda.preferred_linalg_library('magma')

batch = next(iter(val_id_loader))
image = batch["image"].to(device)
features = get_penultimate_features(model, image)

score = calculate_mahalanobis_confidence_score(features, class_means, covariance)
print(score.shape)


torch.Size([48, 48, 48])


In [35]:
import numpy as np

def calculate_scores(loader):
    scores = []
    for batch in tqdm(loader):
        image = batch["image"].to(device)
        if image.ravel().shape[0] == 0:
            print("Empty image encountered, skipping.")
            continue
        features = get_penultimate_features(model, image)
        score = calculate_mahalanobis_confidence_score(features, class_means, covariance)
        scores.append(score.mean().item())
    return scores

val_scores = calculate_scores(val_id_loader)

100%|██████████| 131/131 [00:22<00:00,  5.87it/s]


In [36]:
test_scores = calculate_scores(test_id_loader)
print(test_scores[:10])

100%|██████████| 300/300 [00:35<00:00,  8.42it/s]

[-30.818914435022915, -30.2389850121181, -29.530804411128354, -28.870620914833864, -29.184239018742485, -32.35896773904538, -31.394502259170135, -30.118923865277175, -28.785169635452014, -33.56992996100775]


In [37]:
from dataloading.load_chaos import load_chaos

chaos_loader = load_chaos()

Found 20 Train CT volumes
Found 20 Test CT volumes
Found 0 Train MR volumes
Found 0 Test MR volumes
Total CHAOS volumes: 40
CHAOS dataloader created with 40 samples


In [38]:
chaos_scores = calculate_scores(chaos_loader)
print(chaos_scores[:10])

100%|██████████| 40/40 [01:04<00:00,  1.62s/it]

[-39.9807816418167, -41.19792388328086, -38.97348834527711, -39.348747107800094, -36.32357658915371, -38.15826455994016, -39.91600790821138, -36.723575587292046, -38.366042038316586, -39.02681229764568]


In [39]:
scores = np.concatenate([np.array(chaos_scores), np.array(test_scores)])
labels = np.concatenate([np.ones_like(chaos_scores), np.zeros_like(test_scores)])
labels

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

In [40]:
from sklearn.metrics import average_precision_score
from metrics import fpr

anomaly_scores = -scores

aupr = average_precision_score(labels, anomaly_scores)
fpr90 = fpr(labels, anomaly_scores, percentile=10)
fpr95 = fpr(labels, anomaly_scores, percentile=5)
fpr99 = fpr(labels, anomaly_scores, percentile=1)

print(f"AUPR: {aupr:.4f}, fpr90: {fpr90:.4f}, fpr95: {fpr95:.4f}, fpr99: {fpr99:.4f}")

AUPR: 0.3621, fpr90: 0.1400, fpr95: 0.1500, fpr99: 0.7467


In [41]:
from dataloading.load_brats import load_brats
brats_loader = load_brats()

Found 585 Train BRATS volumes
Found 87 Test BRATS volumes


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


In [42]:
brats_scores = calculate_scores(brats_loader)
print(brats_scores[:10])

  0%|          | 0/672 [00:00<?, ?it/s]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x55bd77e62880): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.00101728

ImageSeriesReader (0x55bd77fe8930): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000545123

  1%|          | 4/672 [00:03<07:43,  1.44it/s]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x55bd77fe8930): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.00100171

  3%|▎         | 22/672 [00:11<03:52,  2.79it/s]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x55bd77e62880): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000668705

ImageSeriesReader (0x55bd76fb11d0): Non uniform sampling or missing slices detected,  maximum

Empty image encountered, skipping.


 33%|███▎      | 220/672 [01:13<02:31,  2.98it/s]


KeyboardInterrupt: 

In [23]:
scores = np.concatenate([np.array(brats_scores), np.array(test_scores)])
labels = np.concatenate([np.ones_like(brats_scores), np.zeros_like(test_scores)])

In [24]:
from sklearn.metrics import average_precision_score
from metrics import fpr

anomaly_scores = -scores

aupr = average_precision_score(labels, anomaly_scores)
fpr90 = fpr(labels, anomaly_scores, percentile=10)
fpr95 = fpr(labels, anomaly_scores, percentile=5)
fpr99 = fpr(labels, anomaly_scores, percentile=1)

print(f"AUPR: {aupr:.4f}, fpr90: {fpr90:.4f}, fpr95: {fpr95:.4f}, fpr99: {fpr99:.4f}")

AUPR: 0.9973, fpr90: 0.0133, fpr95: 0.0433, fpr99: 0.0700
